# 02 — Create the late-delivery label

The historical target uses calendar-date semantics: `late = 1` only when the actual customer delivery calendar date is after the estimated delivery calendar date. Delivery on the estimated date is on time. Rows without an observed actual delivery outcome are inspected and excluded, never assumed on time.

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
assert ROOT.name == "02-olist-late-delivery-ml", f"Run from assignment root or notebooks/, got {ROOT}"
SEED = 42
pd.set_option("display.max_columns", 100)
df=pd.read_parquet(ROOT/"artifacts/01_joined/ml_orders.parquet")
missing=df[df.order_delivered_customer_date.isna()]
missing_status=missing.order_status.value_counts(dropna=False).rename("orders").to_frame()
missing_status

,orders
order_status,
shipped,1107
canceled,619
unavailable,609
invoiced,314
processing,301
delivered,8
created,5
approved,2


In [2]:
eligible=df.order_delivered_customer_date.notna() & df.order_estimated_delivery_date.notna()
labeled=df.loc[eligible].copy()
actual_date = labeled.order_delivered_customer_date.dt.normalize()
estimated_date = labeled.order_estimated_delivery_date.dt.normalize()
labeled["late"] = (actual_date > estimated_date).astype("int8")
assert labeled.late.isin([0,1]).all() and labeled[["order_delivered_customer_date","order_estimated_delivery_date"]].notna().all().all()
counts=labeled.late.value_counts().sort_index(); rates=labeled.late.value_counts(normalize=True).sort_index()
summary={"source_rows":len(df),"unlabeled_rows":int((~eligible).sum()),"labeled_rows":len(labeled),"on_time_count":int(counts[0]),"late_count":int(counts[1]),"on_time_pct":float(rates[0]*100),"late_pct":float(rates[1]*100),"on_time_to_late_ratio":float(counts[0]/counts[1])}
summary

{'source_rows': 99441,
 'unlabeled_rows': 2965,
 'labeled_rows': 96476,
 'on_time_count': 89941,
 'late_count': 6535,
 'on_time_pct': 93.22629462249678,
 'late_pct': 6.773705377503212,
 'on_time_to_late_ratio': 13.762968630451416}

## Manual examples and boundary checks

The examples below are selected from real rows by computed delivery difference: clearly early, clearly late, and closest observed cases to the exact boundary. The assertion recomputes every displayed label.

In [3]:
labeled["delivery_delta_hours"]=(labeled.order_delivered_customer_date-labeled.order_estimated_delivery_date).dt.total_seconds()/3600
examples=pd.concat([labeled.nsmallest(2,"delivery_delta_hours"),labeled.nlargest(2,"delivery_delta_hours"),labeled.iloc[labeled.delivery_delta_hours.abs().argsort()[:4]]]).drop_duplicates("order_id")
examples=examples[["order_id","order_delivered_customer_date","order_estimated_delivery_date","delivery_delta_hours","late"]]
assert (examples.late == (examples.order_delivered_customer_date.dt.normalize() > examples.order_estimated_delivery_date.dt.normalize()).astype(int)).all()
examples

,order_id,order_delivered_customer_date,order_estimated_delivery_date,delivery_delta_hours,late
40094,0607f0efea4b566f1eb8f7d3c2397320,2018-03-09 23:36:47,2018-08-03,-3504.386944,0
15791,c72727d29cde4cf870d569bf65edabfd,2017-02-14 14:27:45,2017-07-04,-3345.537500,0
55619,1b3190b2dfa9d789e1f14c05b647a14a,2018-09-19 23:24:07,2018-03-15,4535.401944,1
19590,ca07593549f1816d26a572e06dc1eab6,2017-09-19 14:36:39,2017-03-22,4358.610833,1
84892,b07a843b1265472b48158269f4de8c36,2018-02-07 23:59:55,2018-02-08,-0.001389,0
44274,2b73cfa36a63e2ba82be54965d188a7e,2018-08-27 23:58:55,2018-08-28,-0.018056,0
3559,9c5b6f458d5e5f8e372c112f8b658af6,2018-03-22 23:58:49,2018-03-23,-0.019722,0
55577,cf3a11ded6899d6b589a1e4a4a4308c0,2018-03-28 23:58:48,2018-03-29,-0.020000,0


## Calendar-date boundary sanity check

The estimate is stored at midnight, while actual delivery includes a time of day. Comparing raw timestamps would incorrectly mark deliveries later on the promised calendar date as late. We therefore normalize both timestamps to midnight before comparison. The check below counts the affected same-date rows and asserts that every one is now on time.

In [4]:
same_calendar_date = (
    labeled.order_delivered_customer_date.dt.normalize()
    == labeled.order_estimated_delivery_date.dt.normalize()
)
same_date_delivered_after_midnight = same_calendar_date & (
    labeled.order_delivered_customer_date > labeled.order_estimated_delivery_date
)
changed_from_timestamp_rule = labeled.loc[same_date_delivered_after_midnight]

assert len(changed_from_timestamp_rule) == 1_292
assert changed_from_timestamp_rule.late.eq(0).all()

boundary_summary = {
    "same_calendar_date_delivered_after_midnight": len(changed_from_timestamp_rule),
    "changed_from_timestamp_late_to_calendar_on_time": len(changed_from_timestamp_rule),
    "percentage_of_labeled": len(changed_from_timestamp_rule) / len(labeled) * 100,
    "all_now_classified_on_time": bool(changed_from_timestamp_rule.late.eq(0).all()),
    "convention": "late only when normalized actual date > normalized estimated date",
}
boundary_summary

{'same_calendar_date_delivered_after_midnight': 1292,
 'changed_from_timestamp_late_to_calendar_on_time': 1292,
 'percentage_of_labeled': 1.339193167212571,
 'all_now_classified_on_time': True,
 'convention': 'late only when normalized actual date > normalized estimated date'}

In [5]:
# Actual delivery and delta remain solely for traceability/EDA and are prohibited from model features.
out=ROOT/"artifacts/02_labeled"; out.mkdir(parents=True,exist_ok=True)
labeled.to_parquet(out/"labeled_orders.parquet",index=False)
(out/"label_summary.json").write_text(json.dumps({**summary,**boundary_summary,"missing_actual_by_status":missing_status.orders.to_dict()},indent=2))
assert (out/"labeled_orders.parquet").exists()
summary

{'source_rows': 99441,
 'unlabeled_rows': 2965,
 'labeled_rows': 96476,
 'on_time_count': 89941,
 'late_count': 6535,
 'on_time_pct': 93.22629462249678,
 'late_pct': 6.773705377503212,
 'on_time_to_late_ratio': 13.762968630451416}